In [1]:
%load_ext autoreload

In [2]:
%autoreload 2
from utils import (
    select_users_by_period,
    create_hourly_user_dataset,
    compute_position_metrics,
)
from visualization_utils import (
    plot_market_features,
    plot_user_metrics,
)
# from visualization_utils import plot_user_metrics



In [3]:
import os
import pandas as pd
from tqdm.auto import tqdm
import re
from datetime import datetime


def add_yield_to_actions(actions_df, yield_df):
    df = actions_df.copy()
    yield_df = yield_df.copy()
    yield_df['timestamp'] = pd.to_datetime(yield_df['timestamp']).astype('int64') // 10**9
    
    # Get min and max timestamps from yield data
    yield_min = yield_df['timestamp'].min()
    yield_max = yield_df['timestamp'].max()
    
    # Filter actions
    df = df[(df['timestamp'] >= yield_min) & (df['timestamp'] <= yield_max + 30 * 24 * 60 * 60)]
    df = df.sort_values('timestamp')
    yield_df = yield_df.sort_values('timestamp')
    df = pd.merge_asof(
        df,
        yield_df[['timestamp', 'base_apy', 'implied_apy', 'underlying_apy']],
        on='timestamp',
        direction='nearest'
    )
    
    return df


from datetime import datetime

def parse_expiry_from_token(token_name):
    parts = token_name.split('-')
    for part in parts:
        if len(part) == 9 and part[:2].isdigit() and part[2:5].isalpha() and part[5:].isdigit():
            # format DDMMMYYYY e.g., 25SEP2025
            return datetime.strptime(part, '%d%b%Y')
    return None
def build_position_statistics(df_actions, token_name):
    expiry = parse_expiry_from_token(token_name)
    if expiry is None:
        raise ValueError(f"Cannot parse expiry from token name: {token_name}")
    expiry_ts = expiry.timestamp()
    
    # Get users with at least one position_open
    open_users = df_actions[df_actions['action'] == 'position_open']['user_address'].unique()
    
    results = []
    for user in open_users:
        user_df = df_actions[df_actions['user_address'] == user].sort_values('timestamp')
        
        # Find first open and first close after that
        open_rows = user_df[user_df['action'] == 'position_open']
        if open_rows.empty:
            continue
        open_time = open_rows.iloc[0]['timestamp']
        open_dt = open_rows.iloc[0]['datetime']
        open_ltv = open_rows.iloc[0]['ltv']
        open_debt = open_rows.iloc[0]['debt']
        
        close_rows = user_df[(user_df['action'] == 'position_close') & (user_df['timestamp'] > open_time)]
        if close_rows.empty:
            continue
        close_time = close_rows.iloc[0]['timestamp']
        close_dt = close_rows.iloc[0]['datetime']
        
        # Subset rows between open and close (inclusive)
        position_df = user_df[(user_df['timestamp'] >= open_time) & (user_df['timestamp'] <= close_time)]
        
        max_debt = position_df['debt'].max()
        max_ltv = position_df['ltv'].max()
        min_ltv = position_df[position_df["ltv"] > 0.1]['ltv'].min()
        
        pos_days = (close_time - open_time) / (24 * 3600)
        days_to_expiry = (expiry_ts - close_time) / (24 * 3600)
        
        results.append({
            'user_address': user,
            'opening_date': open_dt,
            'closing_date': close_dt,
            'position_length_days': pos_days,
            'max_debt': max_debt,
            'max_ltv': max_ltv,
            'min_ltv': min_ltv,
            'initial_ltv': open_ltv,
            'initial_debt': open_debt,
            'days_to_expiry_after_close': days_to_expiry
        })
    
    return pd.DataFrame(results)


def parse_expiry_from_token(token_name):
    # Look for pattern: DDMMMYYYY or DMMMYYYY (where D is digit, MMM is month, YYYY is year)
    match = re.search(r'(\d{1,2}[A-Za-z]{3}\d{4})', token_name)
    if match:
        date_str = match.group(1)
        # Pad day to 2 digits if needed
        if len(date_str) == 8:  # DMMMYYYY format
            date_str = '0' + date_str
        return datetime.strptime(date_str, '%d%b%Y')
    raise ValueError(f"Cannot parse expiry from {token_name}")

def collect_position_metrics_for_all_pt_markets(
    events_dir="/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_enriched",
    hourly_dir="/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_hourly_data",
    yields_dir="/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/pt_yields",
):
    """
    Iterate over all PT-* markets in events_dir, compute position statistics,
    and return a list of dicts with keys: token, market_name, positions_df.
    """
    results = []
    # List all CSV files in events_dir
    for fname in os.listdir(events_dir):
        if not fname.endswith('.csv'):
            continue
        market_name = fname[:-4]  # remove .csv
        # Check if market name starts with "PT-"
        if not market_name.startswith("eth_PT-"):
            continue
        # Extract token name (e.g., PT-USDe-25SEP2025)
        token_name = market_name.split('_')[1]  # assuming format PT-..._usdc
        try:
            expiry = parse_expiry_from_token(token_name)
        except Exception as e:
            continue
        threshold_date = (expiry + pd.Timedelta(days=10)).strftime('%Y-%m-%d')
        # Paths
        events_path = os.path.join(events_dir, fname)
        hourly_path = os.path.join(hourly_dir, f"{market_name}.csv")
        yields_path = os.path.join(yields_dir, f"{token_name}.csv")
        
        # Check if required files exist
        if not os.path.exists(hourly_path):
            print(f"Missing hourly data for {market_name}, skipping")
            continue
        if not os.path.exists(yields_path):
            print(f"Missing yield data for {token_name}, skipping")
            continue
        
        try:
            # Load data
            print(f"Processing market {market_name} (expirity {expiry})... ")
            df = pd.read_csv(events_path)
            market_df = pd.read_csv(hourly_path)
            yield_df = pd.read_csv(yields_path).sort_values("timestamp")
            
            # Add yields to actions
            df = add_yield_to_actions(df, yield_df)
            market_df = add_yield_to_actions(market_df, yield_df)
            
            # Create hourly user dataset
            hourly = create_hourly_user_dataset(
                df,
                market_df,
                n_hours=1,
                threshold_date=threshold_date,
                additional_action_features=["implied_apy"]
            )
            
            # Build position statistics
            pos_stats = build_position_statistics(hourly, token_name)
            
            results.append({
                'token': token_name,
                'market_name': market_name,
                'positions_df': pos_stats,
                'market_df': market_df,
                'raw_df': df,
                'hourly_df': hourly,
            })
            print(f"Processed {market_name}: {len(pos_stats)} positions")
        except Exception as e:
            print(f"Error processing {market_name}: {e}")
            continue
    
    return results

all_markets_position = collect_position_metrics_for_all_pt_markets()

Processing market eth_PT-syrupUSDC-28AUG2025_usdc (expirity 2025-08-28 00:00:00)... 
Processed eth_PT-syrupUSDC-28AUG2025_usdc: 15 positions
Processing market eth_PT-slvlUSD-25SEP2025_usdc (expirity 2025-09-25 00:00:00)... 
Processed eth_PT-slvlUSD-25SEP2025_usdc: 76 positions
Processing market eth_PT-USDe-27NOV2025_usds (expirity 2025-11-27 00:00:00)... 
Processed eth_PT-USDe-27NOV2025_usds: 35 positions
Processing market eth_PT-USDe-25SEP2025_dai (expirity 2025-09-25 00:00:00)... 
Processed eth_PT-USDe-25SEP2025_dai: 128 positions
Processing market eth_PT-syrupUSDC-30OCT2025_usdc (expirity 2025-10-30 00:00:00)... 
Processed eth_PT-syrupUSDC-30OCT2025_usdc: 10 positions
Missing yield data for PT-USDe-26DEC2024, skipping
Processing market eth_PT-stcUSD-23JUL2026_usdc (expirity 2026-07-23 00:00:00)... 
Processed eth_PT-stcUSD-23JUL2026_usdc: 3 positions
Processing market eth_PT-USDe-27NOV2025_usdc (expirity 2025-11-27 00:00:00)... 
Processed eth_PT-USDe-27NOV2025_usdc: 2 positions
Missi

In [6]:
# all_markets_position[0]['positions_df']['closing_date']
# rw = all_markets_position[0]['raw_df']
# openers = []
# suppliers_shares = 
# addrs = all_markets_position[0]['raw_df']['user_address']


import pandas as pd
from datetime import datetime
import re

def parse_expiry_from_token(token_name):
    """Extract expiry date from token name like 'PT-syrupUSDC-28AUG2025'."""
    match = re.search(r'(\d{1,2}[A-Za-z]{3}\d{4})', token_name)
    if match:
        date_str = match.group(1)
        if len(date_str) == 8:
            date_str = '0' + date_str
        return datetime.strptime(date_str, '%d%b%Y')
    return None

def get_base_token(token_name):
    """Extract base token without expiry, e.g., 'PT-syrupUSDC-28AUG2025' -> 'PT-syrupUSDC'."""
    # Remove the last part after last '-' (the expiry date)
    parts = token_name.split('-')
    # Keep all parts except the last one (the expiry)
    return '-'.join(parts[:-1])

def analyze_cross_expiry_positions(market_dicts):
    """
    market_dicts: list of dicts with keys 'token', 'market_name', 'positions_df'
    Returns: dict with base_token -> DataFrame of users with positions in multiple markets.
    """
    # Group by base token
    base_token_dict = {}
    for m in market_dicts:
        token = m['token']
        base = get_base_token(token)
        expiry = parse_expiry_from_token(token)
        if base not in base_token_dict:
            base_token_dict[base] = []
        base_token_dict[base].append({
            'token': token,
            'expiry': expiry,
            'market_name': m['market_name'],
            'positions_df': m['positions_df'].copy()
        })
    
    results = {}
    for base, markets in base_token_dict.items():
        # Sort markets by expiry (earliest first)
        markets.sort(key=lambda x: x['expiry'])
        # For each user, collect positions across markets
        user_positions = {}
        for m in markets:
            df = m['positions_df']
            for _, row in df.iterrows():
                user = row['user_address']
                if user not in user_positions:
                    user_positions[user] = []
                user_positions[user].append({
                    'expiry': m['expiry'],
                    'market_name': m['market_name'],
                    'opening_date': row['opening_date'],
                    'closing_date': row['closing_date'],
                    'max_debt': row['max_debt'],
                    'position_length_days': row['position_length_days'],
                    'days_to_expiry_after_close': row['days_to_expiry_after_close'],
                })
        # Keep only users with positions in more than one market
        multi_market_users = {u: pos for u, pos in user_positions.items() if len(pos) > 1}
        if multi_market_users:
            # Create a DataFrame for easier printing
            records = []
            for user, positions in multi_market_users.items():
                for pos in positions:
                    records.append({
                        'user_address': user,
                        'expiry': pos['expiry'],
                        'market_name': pos['market_name'],
                        'opening_date': pos['opening_date'],
                        'closing_date': pos['closing_date'],
                        'max_debt': pos['max_debt'],
                        'position_length_days': pos['position_length_days'],
                        'days_to_expiry_after_close': pos['days_to_expiry_after_close'],
                    })
            results[base] = pd.DataFrame(records).sort_values(['user_address', 'expiry'])
    
    return results

dd = analyze_cross_expiry_positions(all_markets_position)
for base, df in dd.items():
    print(f"\nBase token: {base}, n markets {df['market_name'].nunique()}")
    df['user_max_debt'] = df.groupby('user_address')['max_debt'].transform('max')

    # Sort by user_max_debt descending, then by expiry
    df_sorted = df.sort_values(['user_max_debt', 'expiry'], ascending=[False, True])

    # Drop the helper column
    df_sorted = df_sorted.drop(columns=['user_max_debt'])

    # Print
    print(df_sorted.to_string(index=False))




Base token: PT-slvlUSD, n markets 2
                              user_address     expiry                   market_name        opening_date        closing_date     max_debt  position_length_days  days_to_expiry_after_close
0xC28e8f073439800dF7A4934d198E1966B4Dd8A25 2025-05-29 eth_PT-slvlUSD-29MAY2025_usdc 2025-04-08 07:03:59 2025-05-29 07:03:59 6.397240e+06             51.000000                   -0.419433
0xC28e8f073439800dF7A4934d198E1966B4Dd8A25 2025-09-25 eth_PT-slvlUSD-25SEP2025_usdc 2025-05-29 06:38:47 2025-07-21 03:38:47 1.019801e+06             52.875000                   65.723067
0xFbbF1826Aba90704A2167D2EB4A7a9D83A8DE9c7 2025-05-29 eth_PT-slvlUSD-29MAY2025_usdc 2025-04-22 17:16:23 2025-05-29 21:16:23 6.395783e+06             37.166667                   -1.011377
0xFbbF1826Aba90704A2167D2EB4A7a9D83A8DE9c7 2025-09-25 eth_PT-slvlUSD-25SEP2025_usdc 2025-05-21 20:47:47 2025-07-19 23:47:47 1.137468e+06             59.125000                   66.883484
0x37829FE9b8e67B8267C2058b94

In [8]:
for base, df in dd.items():
    print(f"\nBase token: {base}, n markets {df['market_name'].nunique()}")
    same_expirity_users = df.groupby(["user_address", 'expiry'])['market_name'].count()
    same_expirity_users = same_expirity_users[same_expirity_users>1].reset_index()
    df = df[df['user_address'].isin(same_expirity_users['user_address']) & df['expiry'].isin(same_expirity_users['expiry'])]
    df['user_max_debt'] = df.groupby('user_address')['max_debt'].transform('max')

    # Sort by user_max_debt descending, then by expiry
    df_sorted = df.sort_values(['user_max_debt', 'expiry'], ascending=[False, True])

    # Drop the helper column
    df_sorted = df_sorted.drop(columns=['user_max_debt'])

    # Print
    print(df_sorted.to_string(index=False))




Base token: PT-slvlUSD, n markets 2
Empty DataFrame
Columns: [user_address, expiry, market_name, opening_date, closing_date, max_debt, position_length_days, days_to_expiry_after_close]
Index: []

Base token: PT-USDe, n markets 7
                              user_address     expiry                market_name        opening_date        closing_date     max_debt  position_length_days  days_to_expiry_after_close
0xDbc652411605f8bB8969136923aa039c79989482 2025-09-25  eth_PT-USDe-25SEP2025_dai 2025-08-26 02:35:47 2025-09-13 19:35:47 7.992265e+07             18.708333                   11.058484
0xDbc652411605f8bB8969136923aa039c79989482 2025-09-25 eth_PT-USDe-25SEP2025_usdc 2025-08-22 18:18:47 2025-09-13 19:18:47 6.678592e+07             22.041667                   11.070289
0xDbc652411605f8bB8969136923aa039c79989482 2025-11-27 eth_PT-USDe-27NOV2025_usds 2025-09-13 18:21:59 2025-09-18 12:21:59 1.127210e+08              4.750000                   69.359734
0xf2085a1521D11396FD182C7a5Fb2622f

/var/folders/hj/pbs977kd43s6n1l9z3mxrj200000gn/T/ipykernel_16396/1928309420.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['user_max_debt'] = df.groupby('user_address')['max_debt'].transform('max')
/var/folders/hj/pbs977kd43s6n1l9z3mxrj200000gn/T/ipykernel_16396/1928309420.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['user_max_debt'] = df.groupby('user_address')['max_debt'].transform('max')


In [60]:
for i in range(len(all_markets_position)):
    if all_markets_position[i]['raw_df']['implied_apy'].max() == 0:
        print(all_markets_position[i]['market_name'])


In [29]:
from collections import defaultdict
base_to_index = defaultdict(list)
market2index = {}
for i in range(len(all_markets_position)):
    market2index[all_markets_position[i]['market_name']] = i
    base_to_index[get_base_token(all_markets_position[i]['token'])].append(i)
base_to_index
market2index

{'eth_PT-syrupUSDC-28AUG2025_usdc': 0,
 'eth_PT-slvlUSD-25SEP2025_usdc': 1,
 'eth_PT-USDe-27NOV2025_usds': 2,
 'eth_PT-USDe-25SEP2025_dai': 3,
 'eth_PT-syrupUSDC-30OCT2025_usdc': 4,
 'eth_PT-stcUSD-23JUL2026_usdc': 5,
 'eth_PT-USDe-27NOV2025_usdc': 6,
 'eth_PT-wstUSR-27MAR2025_usr': 7,
 'eth_PT-sNUSD-5MAR2026_usdc': 8,
 'eth_PT-USD0++-31OCT2024_usdc': 9,
 'eth_PT-USDe-31JUL2025_dai': 10,
 'eth_PT-USDe-25SEP2025_usdt': 11,
 'eth_PT-USDe-25SEP2025_usdc': 12,
 'eth_PT-wstUSR-27MAR2025_usdc': 13,
 'eth_PT-slvlUSD-29MAY2025_usdc': 14,
 'eth_PT-mHYPER-20NOV2025_usdc': 15,
 'eth_PT-reUSD-25JUN2026_usdc': 16,
 'eth_PT-reUSD-18DEC2025_usdc': 17,
 'eth_PT-RLP-4SEP2025_usdc': 18,
 'eth_PT-csUSDL-31JUL2025_usdc': 19,
 'eth_PT-wstUSR-25SEP2025_usdc': 20,
 'eth_PT-stcUSD-29JAN2026_usdc': 21,
 'eth_PT-lvlUSD-29MAY2025_usdc': 22,
 'eth_PT-USR-29MAY2025_usdc': 23,
 'eth_PT-USDe-27MAR2025_dai': 24,
 'eth_PT-USD0++-27MAR2025_usdc': 25}

In [26]:
all_markets_position[0]['market_df']

,timestamp,datetime,total_supply,total_borrow,utilization,borrow_rate,supply_rate,volatility_1h,drawdown_1h,volatility_6h,drawdown_6h,collateral_price,loan_asset_price,avg_health_factor,borrow_rate_rolling,supply_rate_rolling,asset_price,base_apy,implied_apy,underlying_apy
0,1747184400,2025-05-14 01:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.976228,0.102539,0.090325,0.069096
1,1747188000,2025-05-14 02:00:00,1.000000,1.000000,1.000000,0.159420,0.159420,0,0,0.000116,0.0,0.975220,1.000000,1.829326,0.079710,0.079710,0.975220,0.102181,0.090325,0.069088
2,1747191600,2025-05-14 03:00:00,1.000000,1.000000,1.000000,0.159420,0.159420,0,0,0.000116,0.0,0.975220,1.000000,1.829326,0.106280,0.106280,0.975229,0.102219,0.090325,0.069081
3,1747195200,2025-05-14 04:00:00,1.000000,1.000000,1.000000,0.159420,0.159420,0,0,0.000116,0.0,0.975220,1.000000,1.829326,0.119565,0.119565,0.975229,0.102236,0.090325,0.069073
4,1747198800,2025-05-14 05:00:00,1.000000,1.000000,1.000000,0.159420,0.159420,0,0,0.000116,0.0,0.975220,1.000000,1.829326,0.127536,0.127536,0.975248,0.102602,0.090325,0.069064
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3258,1758913200,2025-09-26 19:00:00,59.877178,28.977748,0.483953,0.015688,0.007593,0,0,0.000007,0.0,0.999789,0.999803,0.000000,0.015688,0.007593,0.999799,0.088353,0.105727,0.069521
3259,1758916800,2025-09-26 20:00:00,59.877178,28.977748,0.483953,0.015688,0.007593,0,0,0.000007,0.0,0.999789,0.999803,0.000000,0.015688,0.007593,0.999710,0.088353,0.105727,0.069521
3260,1758920400,2025-09-26 21:00:00,59.877178,28.977748,0.483953,0.015688,0.007593,0,0,0.000007,0.0,0.999789,0.999803,0.000000,0.015688,0.007593,0.999621,0.088353,0.105727,0.069521
3261,1758924000,2025-09-26 22:00:00,59.877178,28.977748,0.483953,0.015688,0.007593,0,0,0.000007,0.0,0.999789,0.999803,0.000000,0.015688,0.007593,0.999621,0.088353,0.105727,0.069521


In [61]:
# for ind in base_to_index['PT-USDe']:
# for ind in [3,11,12,]:
for ind in [2,]:
    print(all_markets_position[ind]['market_name'])
    all_markets_position[ind]['market_df']['rate_spread'] = all_markets_position[ind]['market_df']['implied_apy'] - all_markets_position[ind]['market_df']['borrow_rate']
    plot_market_features(all_markets_position[ind]['market_df'], ['implied_apy', 'borrow_rate'], y_ranges={"borrow_rate": [0,0.3]}, dates_range=['2025-07-01', '2025-09-25'])

eth_PT-USDe-27NOV2025_usds


In [66]:
import numpy as np
def add_position_features(positions_df, market_df):
    """
    Add features to positions DataFrame using hourly market data.

    Parameters:
    positions_df : DataFrame with columns 'opening_date', 'closing_date' (datetime)
    market_df    : DataFrame with columns 'timestamp', 'datetime', 'borrow_rate', 'implied_apy'

    Returns:
    positions_df with additional columns:
        - initial_implied_apy
        - close_implied_apy
        - implied_apy_diff
        - eff_borrow_rate_overall
        - eff_borrow_rate_last_24h
        - eff_borrow_rate_last_72h
        - spread_overall, spread_last_24h, spread_last_72h
    """
    df_pos = positions_df.copy()
    df_market = market_df.copy()

    # Convert dates to timestamps if not already
    if 'opening_date' in df_pos.columns:
        df_pos['open_ts'] = pd.to_datetime(df_pos['opening_date']).astype('int64') // 10**9
    if 'closing_date' in df_pos.columns:
        df_pos['close_ts'] = pd.to_datetime(df_pos['closing_date']).astype('int64') // 10**9

    # Market data: ensure sorted
    df_market = df_market.sort_values('timestamp')
    market_ts = df_market['timestamp'].values
    market_borrow = df_market['borrow_rate'].values
    market_apy = df_market['implied_apy'].values

    # 1. Get implied APY at open and close (using last available value ≤ timestamp)
    df_market_apy = df_market[['timestamp', 'implied_apy']].dropna()
    # Open APY
    df_pos = df_pos.sort_values('open_ts')
    df_pos = pd.merge_asof(
        df_pos,
        df_market_apy,
        left_on='open_ts',
        right_on='timestamp',
        direction='backward'
    ).rename(columns={'implied_apy': 'initial_implied_apy'}).drop(columns='timestamp')
    # Close APY
    df_pos = df_pos.sort_values('close_ts')
    df_pos = pd.merge_asof(
        df_pos,
        df_market_apy,
        left_on='close_ts',
        right_on='timestamp',
        direction='backward'
    ).rename(columns={'implied_apy': 'close_implied_apy'}).drop(columns='timestamp')
    # Restore original order
    df_pos = df_pos.sort_index()

    # Compute APY change
    df_pos['implied_apy_diff'] = df_pos['close_implied_apy'] - df_pos['initial_implied_apy']

    # 2. Compute effective borrow rates
    pos_opens = df_pos['open_ts'].values
    pos_closes = df_pos['close_ts'].values
    n_pos = len(df_pos)
    eff_overall = np.full(n_pos, np.nan)
    eff_last24 = np.full(n_pos, np.nan)
    eff_last72 = np.full(n_pos, np.nan)

    for i in range(n_pos):
        open_ts = pos_opens[i]
        close_ts = pos_closes[i]
        if np.isnan(open_ts) or np.isnan(close_ts):
            continue

        # Overall period
        left = np.searchsorted(market_ts, open_ts, side='left')
        right = np.searchsorted(market_ts, close_ts, side='right')
        if left < right:
            window_rates = market_borrow[left:right]
            eff_overall[i] = window_rates.mean()

        # Last 24h before close
        start24 = close_ts - 24 * 3600
        left24 = np.searchsorted(market_ts, start24, side='left')
        right24 = np.searchsorted(market_ts, close_ts, side='right')
        if left24 < right24:
            window24 = market_borrow[left24:right24]
            eff_last24[i] = window24.mean()

        # Last 72h before close
        start72 = close_ts - 72 * 3600
        left72 = np.searchsorted(market_ts, start72, side='left')
        right72 = np.searchsorted(market_ts, close_ts, side='right')
        if left72 < right72:
            window72 = market_borrow[left72:right72]
            eff_last72[i] = window72.mean()

    df_pos['eff_borrow_rate_overall'] = eff_overall
    df_pos['eff_borrow_rate_last_24h'] = eff_last24
    df_pos['eff_borrow_rate_last_72h'] = eff_last72

    # 3. Spreads (initial implied APY minus effective borrow rates)
    df_pos['spread_overall'] = df_pos['initial_implied_apy'] - df_pos['eff_borrow_rate_overall']
    df_pos['spread_last_24h'] = df_pos['initial_implied_apy'] - df_pos['eff_borrow_rate_last_24h']
    df_pos['spread_last_72h'] = df_pos['initial_implied_apy'] - df_pos['eff_borrow_rate_last_72h']

    # Drop helper columns
    df_pos.drop(columns=['open_ts', 'close_ts'], inplace=True)

    return df_pos
print(all_markets_position[0].keys())
for m in range(len(all_markets_position)):
    all_markets_position[m]['early_closing'] = all_markets_position[m]['positions_df'][all_markets_position[m]['positions_df']['days_to_expiry_after_close'] > 7]
    all_markets_position[m]['late_closing'] = all_markets_position[m]['positions_df'][all_markets_position[m]['positions_df']['days_to_expiry_after_close'] <= 7]
    all_markets_position[m]['early_closing'] = add_position_features(all_markets_position[m]['early_closing'], all_markets_position[m]['market_df'])
    all_markets_position[m]['late_closing'] = add_position_features(all_markets_position[m]['late_closing'], all_markets_position[m]['market_df'])
    all_markets_position[m]['positions_df_enriched'] = add_position_features(all_markets_position[m]['positions_df'], all_markets_position[m]['market_df'])
    print(all_markets_position[m]['market_name'], len(all_markets_position[m]['positions_df']), len(all_markets_position[m]['early_closing']))

dict_keys(['token', 'market_name', 'positions_df', 'market_df', 'raw_df', 'hourly_df', 'early_closing', 'positions_df_enriched', 'late_closing'])
eth_PT-syrupUSDC-28AUG2025_usdc 15 8
eth_PT-slvlUSD-25SEP2025_usdc 76 41
eth_PT-USDe-27NOV2025_usds 35 29
eth_PT-USDe-25SEP2025_dai 128 29
eth_PT-syrupUSDC-30OCT2025_usdc 10 3
eth_PT-stcUSD-23JUL2026_usdc 3 3
eth_PT-USDe-27NOV2025_usdc 2 2
eth_PT-wstUSR-27MAR2025_usr 51 9
eth_PT-sNUSD-5MAR2026_usdc 6 2
eth_PT-USD0++-31OCT2024_usdc 23 1
eth_PT-USDe-31JUL2025_dai 95 71
eth_PT-USDe-25SEP2025_usdt 116 55
eth_PT-USDe-25SEP2025_usdc 216 84
eth_PT-wstUSR-27MAR2025_usdc 10 10
eth_PT-slvlUSD-29MAY2025_usdc 35 13
eth_PT-mHYPER-20NOV2025_usdc 77 51
eth_PT-reUSD-25JUN2026_usdc 223 223
eth_PT-reUSD-18DEC2025_usdc 21 7
eth_PT-RLP-4SEP2025_usdc 63 28
eth_PT-csUSDL-31JUL2025_usdc 107 33
eth_PT-wstUSR-25SEP2025_usdc 60 45
eth_PT-stcUSD-29JAN2026_usdc 129 98
eth_PT-lvlUSD-29MAY2025_usdc 85 24
eth_PT-USR-29MAY2025_usdc 18 5
eth_PT-USDe-27MAR2025_dai 122 73
eth_

In [67]:
all_markets_position[0]['early_closing'].head()

,user_address,opening_date,closing_date,position_length_days,max_debt,max_ltv,min_ltv,initial_ltv,initial_debt,days_to_expiry_after_close,initial_implied_apy,close_implied_apy,implied_apy_diff,eff_borrow_rate_overall,eff_borrow_rate_last_24h,eff_borrow_rate_last_72h,spread_overall,spread_last_24h,spread_last_72h
0,0x594c5eE5a326295FF6212C31BB3311747D3bD562,2025-05-14 01:55:59,2025-05-19 09:55:59,5.333333,1.000000,0.500184,0.499441,0.000000,1.000000,100.461123,0.090325,0.089479,-8.457159e-04,0.156045,0.141420,0.153420,-0.065720,-0.051095,-0.063095
1,0xABfE00f81c2b9734C2FeCec3f1996E18611ce658,2025-05-20 11:26:59,2025-05-20 12:26:59,0.041667,149.968800,0.511602,0.511602,0.511602,149.968800,99.356262,0.088595,0.088595,4.503106e-12,0.096055,0.096929,0.133086,-0.007460,-0.008334,-0.044491
2,0xE126E7c4A06D93a6e07e0DBb97a2027362e856c3,2025-05-31 06:59:47,2025-06-06 11:59:47,6.208333,36746.764781,0.892149,0.837196,0.837196,22959.627354,82.375150,0.092288,0.099253,6.965582e-03,0.096903,0.158871,0.112556,-0.004616,-0.066583,-0.020268
3,0x5fF11eAc5a8ffE52FBeA6a1431f7795920E10D09,2025-06-04 19:53:23,2025-06-11 18:53:23,6.958333,21513.132461,0.803133,0.801416,0.803132,21513.132461,77.087928,0.098063,0.100229,2.166367e-03,0.105411,0.091128,0.095629,-0.007348,0.006935,0.002434
4,0x6B84C4cBa94bd0971694f7f122635503De96112f,2025-06-11 04:39:47,2025-06-25 02:39:47,13.916667,24981.423873,0.838084,0.832786,0.838084,24981.423873,63.764039,0.096347,0.088044,-8.302946e-03,0.100251,0.080989,0.085828,-0.003904,0.015358,0.010518


In [74]:
users_clusters

,user_address,cluster
0,0x002AFa1e35cE85272aAE5e0d7Ab541DFd442B667,2
1,0x00896485a15c549AA6948c7430bef1f02f752300,3
2,0x008E5d9ea233Dfb9AB48098d6E98E14374030c15,-1
3,0x00A2913501C4B09B92b825dC8A2937eFDaD9953b,0
4,0x00B8dF76c223eb4b05123389330b4afD157152b8,0
...,...,...
1617,0xfc9803f19a667Ac43E13fC57405f9fDCdaC54747,2
1618,0xfcB2f00cF544E934E2f163c2593c7CE3eF7fFA9D,2
1619,0xfd1fA2C22F65649AE02188a33F286f22bcFCd225,2
1620,0xff17d5A9f7dB38A1384CF0ce1c81b880D17dB838,2


In [75]:
users_clusters = pd.read_csv("/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/modelling_datasets/pt_users_clusters_mapping.csv")
all_markets_position[0]['early_closing']
for i in all_markets_position:
    if '2026' in i['market_name']:
        continue
    print(i['market_name'])
    
    early = i['early_closing']
    late = i['late_closing']
    
    # Early closing
    early_pos = early[early['spread_overall'] > 0]
    early_neg = early[early['spread_overall'] < 0]
    early_neg_72 = early[early['spread_last_72h'] < 0]
    market_max_debt = i['market_df']['total_borrow'].quantile(0.95)
    print(f"    Early ({len(early)} events):")
    print(f"        Positive spread: {len(early_pos)} events, mean overall {early_pos['spread_overall'].mean():.4f}, 24h {early_pos['spread_last_24h'].mean():.4f}, 72h {early_pos['spread_last_72h'].mean():.4f}, implied apy diff {early_pos['implied_apy_diff'].mean()}, debt {early_pos['max_debt'].mean()} (pct. {early_pos['max_debt'].mean() / market_max_debt})")
    print(f"        Negative spread: {len(early_neg)} events, mean overall {early_neg['spread_overall'].mean():.4f}, 24h {early_neg['spread_last_24h'].mean():.4f}, 72h {early_neg['spread_last_72h'].mean():.4f}, implied apy diff {early_neg['implied_apy_diff'].mean()}, debt {early_neg['max_debt'].mean()} (pct. {early_neg['max_debt'].mean() / market_max_debt})")
    print(f"        Negative spread 72h: {len(early_neg_72)} events, mean overall {early_neg_72['spread_overall'].mean():.4f}, 24h {early_neg_72['spread_last_24h'].mean():.4f}, 72h {early_neg_72['spread_last_72h'].mean():.4f}, implied apy diff {early_neg_72['implied_apy_diff'].mean()},  debt {early_neg_72['max_debt'].mean()} (pct. {early_neg_72['max_debt'].mean() / market_max_debt})")
    print(f"Early clusters {users_clusters[users_clusters['user_address'].isin(early_pos['user_address'].unique())]['cluster'].value_counts()}")
    # Late closing
    late_neg = late[late['spread_overall'] < 0]
    print(f"    Late ({len(late)} events): negative spread users {len(late_neg)}")


eth_PT-syrupUSDC-28AUG2025_usdc
    Early (8 events):
        Positive spread: 1 events, mean overall 0.0007, 24h -0.0008, 72h 0.0078, implied apy diff -0.01231514652578869, debt 15713.09962221284 (pct. 0.0022386332801133235)
        Negative spread: 7 events, mean overall -0.0151, 24h -0.0237, 72h -0.0230, implied apy diff -0.0011730103499520872, debt 983898.6819539367 (pct. 0.14017529237630655)
        Negative spread 72h: 5 events, mean overall -0.0189, 24h -0.0376, 72h -0.0348, implied apy diff -0.0004148986175256414,  debt 1368159.243468761 (pct. 0.19492060055382676)
Early clusters cluster
2    1
Name: count, dtype: int64
    Late (7 events): negative spread users 0
eth_PT-slvlUSD-25SEP2025_usdc
    Early (41 events):
        Positive spread: 40 events, mean overall 0.0213, 24h 0.0121, 72h 0.0155, implied apy diff 0.0024994478434475177, debt 167479.7002485108 (pct. 0.01829491901773462)
        Negative spread: 1 events, mean overall -0.0009, 24h 0.0046, 72h 0.0039, implied apy dif